# 04 BMTC population coverage (Bengaluru)

Question (T041): what share of Bangalore district's population lives within a 500 m **walk**
of a BMTC bus stop with scheduled weekday service?

Inputs, all local and versioned: S30 BMTC GTFS (unofficial Vonter feed, commit ee945dc, 8 Sep
2026; weekday 9 Sep 2026), S18 OSM walking network (owner download, D174), S16 WorldPop 2025
1 km population, S17 Bangalore district boundary. Method: `src/atlas/analysis/coverage.py`.

Each person is counted once: a 1 km cell is covered in proportion to its walkable length within
500 m of a stop along the network, so overlapping catchments never add population twice, and
covered plus uncovered equals the district total. This measures scheduled access, not service
quality, crowding or safety.


In [ ]:
import json
import os
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from atlas.analysis.coverage import run
from atlas.analysis.quality import caption

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/04_bmtc_coverage"))
result = run(OUT)
summary = result["summary"]
print(json.dumps(summary["population"], indent=1))
print(json.dumps(summary["stops"], indent=1))

In [ ]:
cells = result["cells"]
centre = result["centre"]
xs = [centre[c["cell_id"]][0] for c in cells]
ys = [centre[c["cell_id"]][1] for c in cells]
frac = [c["covered_fraction"] for c in cells]
pop = [c["population"] for c in cells]
cap = caption(
    "S16 WorldPop 2025, S18 OSM, S30 BMTC GTFS (unofficial)",
    "share of each 1 km cell's walkable length within 500 m of a served stop",
    "fraction",
    summary["service_date"],
    summary["service_date"],
    f"{summary['population']['cells']} cells, Bangalore district",
)
fig, ax = plt.subplots(figsize=(7, 7))
sc = ax.scatter(
    xs, ys, c=frac, s=[max(4, p / 2000) for p in pop], cmap="viridis", vmin=0, vmax=1, marker="s"
)
fig.colorbar(sc, ax=ax, label="covered fraction (marker size: population)")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_aspect("equal")
ax.set_title("BMTC 500 m walking coverage by 1 km cell")
fig.text(0.01, 0.01, cap, fontsize=6, wrap=True)
fig.savefig(OUT / "coverage_map.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
hours = result["hours"]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(list(hours), list(hours.values()), color="#3b6ea5")
ax.set_xlabel("service hour of first departure")
ax.set_ylabel("trips starting")
ax.set_title(f"BMTC scheduled trips per hour, {summary['service_date']} (unofficial GTFS)")
fig.text(
    0.01,
    0.01,
    caption(
        "S30 BMTC GTFS (Vonter, unofficial)",
        "trips by first departure hour",
        "trips",
        summary["service_date"],
        summary["service_date"],
        f"{summary['trips']['service_day_trips']} trips",
    ),
    fontsize=6,
)
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(OUT / "trips_per_hour.png", dpi=80, metadata={"Software": None})
plt.close(fig)
print(summary["trips"])

## Limits

Read `summary.json` field `limits` with every number above: scheduled supply from an unofficial
feed; OSM walkway completeness unknown (population in cells without a mapped walkway counts as
uncovered); WorldPop is modelled; stops outside the district are not used; access only, no
frequency, crowding, fare or safety judgement.